# Variant 12 — Quality-40k Mixed Answer + Short-Solution LoRA

Notebook này là biến thể model-only dựa trên Variant 11, nhưng thay đổi hai điểm chính:

- Từ full `train.json`, lọc và chọn **40k mẫu chất lượng** thay vì shuffle ngẫu nhiên.
- Train mixed target: phần lớn **answer-only** để giữ đáp án ổn định, một phần **short-solution** để model học sinh lời giải ngắn kèm anchor `Đáp án là:`.

Không dùng rule solver, không dùng Python để tự giải toán từ `query_vi`, không thay base model, không dùng test-time learning.


## Cell 1 — Setup, config, paths


In [1]:
import os
import re
import gc
import json
import math
import random
import inspect
import warnings
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

VARIANT_NAME = "v12_quality40k_mixed_solution_lora_model_only"
SAFE_EOS_ID = 50256

# Kaggle official paths. Local fallback paths are included only for debugging outside Kaggle.
TRAIN_PATH_CANDIDATES = [
    Path("/kaggle/input/datasets/kimanh2002/dataset-math/train.json"),
    Path("/kaggle/input/dataset-math/train.json"),
    Path("/mnt/data/train.json"),
]
VALID_PATH_CANDIDATES = [
    Path("/kaggle/input/datasets/kimanh2002/dataset-math/valid.json"),
    Path("/kaggle/input/dataset-math/valid.json"),
    Path("/mnt/data/valid.json"),
]
TEST_PATH_CANDIDATES = [
    Path("/kaggle/input/datasets/kimanh2002/dataset-math/test.json"),
    Path("/kaggle/input/dataset-math/test.json"),
    Path("/mnt/data/test.json"),
]
MODEL_PATH_CANDIDATES = [
    Path("/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese"),
    Path("/kaggle/input/nlphustgpt2-vietnamese"),
    Path("/kaggle/input/nlp-hust-gpt2-vietnamese"),
    Path("/kaggle/input/gpt2-vietnamese"),
]

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/mnt/data")
PROC_DIR = WORK_DIR / f"processed_{VARIANT_NAME}"
MODEL_OUT_DIR = WORK_DIR / f"lora_{VARIANT_NAME}"
PRED_DIR = WORK_DIR / f"predictions_{VARIANT_NAME}"
for p in [PROC_DIR, MODEL_OUT_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Main knobs for <= 3 hours.
# This variant keeps exactly 40k selected train examples, but mixes target types.
TRAIN_MAX_SAMPLES = 40000
SOLUTION_TARGET_RATIO = 0.35
MAX_SOLUTION_CHARS = 650
MAX_LENGTH = 384

# Inference generation lengths.
ANSWER_GEN_MAX_NEW_TOKENS = 24
SOLUTION_GEN_MAX_NEW_TOKENS = 96
GEN_MAX_NEW_TOKENS = SOLUTION_GEN_MAX_NEW_TOKENS  # kept for compatibility in reports

# Sampling is disabled by default because numeric tasks are usually more stable with greedy decoding.
USE_SAMPLING_FALLBACK = False
NUM_SAMPLE_CANDIDATES = 2
GEN_TEMPERATURE = 0.6
GEN_TOP_P = 0.9

NUM_TRAIN_EPOCHS = 4
PER_DEVICE_TRAIN_BATCH_SIZE = 8
PER_DEVICE_EVAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

LOGGING_STEPS = 50
EVAL_STEPS = 700
SAVE_STEPS = 700
EVAL_LOSS_MAX_SAMPLES = 512
VALID_EVAL_MAX_SAMPLES = None  # None = full valid; set 300/500 for quick smoke test.

DO_TRAIN = True
DO_VALIDATE = True
DO_TEST_PREDICT = True

print("Variant:", VARIANT_NAME)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Variant: v7_answer_only_lora_rule_solver
CUDA: True
GPU: Tesla T4


## Cell 2 — I/O helpers and path discovery

In [2]:

def first_existing(paths: List[Path], required: bool = True, name: str = "path") -> Optional[Path]:
    for p in paths:
        if p.exists():
            print(f"{name}: {p}")
            return p
    if required:
        raise FileNotFoundError(f"Cannot find {name}. Tried: {[str(p) for p in paths]}")
    print(f"{name}: not found")
    return None

TRAIN_PATH = first_existing(TRAIN_PATH_CANDIDATES, True, "TRAIN_PATH")
VALID_PATH = first_existing(VALID_PATH_CANDIDATES, True, "VALID_PATH")
TEST_PATH = first_existing(TEST_PATH_CANDIDATES, False, "TEST_PATH")


def find_model_path() -> Path:
    for p in MODEL_PATH_CANDIDATES:
        if p.exists():
            print("MODEL_PATH:", p)
            return p
    # Fallback: search common Kaggle input subdirs for config.json.
    root = Path("/kaggle/input")
    if root.exists():
        for cfg in root.rglob("config.json"):
            parent = cfg.parent
            files = {x.name for x in parent.iterdir() if x.is_file()}
            if any(name.startswith("pytorch_model") or name.endswith(".safetensors") for name in files):
                print("MODEL_PATH auto-found:", parent)
                return parent
    raise FileNotFoundError("Cannot find local NlpHUST/gpt2-vietnamese model directory.")


def read_json_or_jsonl(path: Path) -> Any:
    text = path.read_text(encoding="utf-8-sig").strip()
    if not text:
        return []
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # JSONL fallback.
    records = []
    ok_jsonl = True
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            ok_jsonl = False
            break
    if ok_jsonl and records:
        return records

    # Concatenated JSON objects fallback.
    decoder = json.JSONDecoder()
    idx = 0
    records = []
    while idx < len(text):
        while idx < len(text) and text[idx].isspace():
            idx += 1
        if idx >= len(text):
            break
        obj, end = decoder.raw_decode(text, idx)
        records.append(obj)
        idx = end
    return records


def ensure_list_records(obj: Any) -> List[Dict[str, Any]]:
    if isinstance(obj, list):
        return [x for x in obj if isinstance(x, dict)]
    if isinstance(obj, dict):
        for key in ["data", "records", "items", "examples"]:
            if isinstance(obj.get(key), list):
                return [x for x in obj[key] if isinstance(x, dict)]
        return [obj]
    return []


def write_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def write_jsonl(records: List[Dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

raw_train = ensure_list_records(read_json_or_jsonl(TRAIN_PATH))
raw_valid = ensure_list_records(read_json_or_jsonl(VALID_PATH))
print("raw_train:", len(raw_train), "raw_valid:", len(raw_valid))
print("train sample keys:", raw_train[0].keys() if raw_train else None)


TRAIN_PATH: /kaggle/input/datasets/kimanh2002/dataset-math/train.json
VALID_PATH: /kaggle/input/datasets/kimanh2002/dataset-math/valid.json
TEST_PATH: not found
raw_train: 95400 raw_valid: 1000
train sample keys: dict_keys(['original_question_vi', 'original_question_en', 'query_vi', 'query_en', 'response_vi', 'response_en', 'type'])


## Cell 3 — Answer extraction and scoring

Extractor này ưu tiên đáp án sau anchor (`Đáp án là`, `Câu trả lời là`, `The answer is`, `####`) và xử lý cơ bản `\frac{a}{b}`, `a/b`, `a\pi`, decimal comma, thousand separator.

In [3]:

ANSWER_ANCHOR_PATTERNS = [
    r"Đáp án là\s*[:：]?\s*([^\n]+)",
    r"Câu trả lời là\s*[:：]?\s*([^\n]+)",
    r"The answer is\s*[:：]?\s*([^\n]+)",
    r"####\s*([^\n]+)",
]

BAD_ANSWER_TOKENS = ["nan", "inf", "undefined", "không xác định", "không đủ", "unknown", "none", "n/a"]


def normalize_unicode_text(text: str) -> str:
    text = str(text or "")
    text = text.replace("\ufeff", "").replace("\u200b", "").replace("\u00a0", " ")
    text = text.replace("−", "-").replace("–", "-").replace("—", "-")
    text = text.replace("×", "*").replace("÷", "/")
    text = text.replace("\\đóng hộp", "\\boxed").replace("\\ đóng hộp", "\\boxed")
    text = text.replace("\\dfrac", "\\frac").replace("\\tfrac", "\\frac")
    return text


def strip_asy_blocks(text: str) -> str:
    return re.sub(r"\[asy\].*?\[/asy\]", " ", str(text or ""), flags=re.I | re.S)


def unwrap_boxed_once(s: str) -> str:
    m = re.search(r"\\boxed\s*\{", s)
    if not m:
        return s
    start = m.end()
    depth = 1
    i = start
    while i < len(s):
        if s[i] == "{":
            depth += 1
        elif s[i] == "}":
            depth -= 1
            if depth == 0:
                return s[:m.start()] + s[start:i] + s[i+1:]
        i += 1
    return s


def strip_boxed(s: str) -> str:
    s = str(s or "")
    for _ in range(5):
        s2 = unwrap_boxed_once(s)
        if s2 == s:
            break
        s = s2
    return s


def clean_answer_candidate(ans: str) -> str:
    ans = normalize_unicode_text(ans)
    ans = strip_boxed(ans).strip()
    ans = ans.replace("$", "").replace("`", "").strip()
    ans = re.split(r"\s+(?:Đáp án là|Câu trả lời là|The answer is)\s*[:：]?", ans, flags=re.I)[0]
    ans = ans.split("\n")[0].strip()
    ans = re.sub(r"^[=:：\s]+", "", ans).strip()
    ans = re.sub(r"(?:\.|,|;|:|。)+$", "", ans).strip()
    ans = re.sub(r"\s+", " ", ans)

    # Prefer latex fraction if present.
    m = re.search(r"-?\\frac\s*\{\s*-?\d+(?:[\.,]\d+)?\s*\}\s*\{\s*-?\d+(?:[\.,]\d+)?\s*\}", ans)
    if m:
        return m.group(0).replace(" ", "")

    # Prefer tuple if present.
    m = re.search(r"\((-?\d+(?:[\.,]\d+)?\s*,\s*)+-?\d+(?:[\.,]\d+)?\)", ans)
    if m:
        return re.sub(r"\s+", "", m.group(0))

    # Prefer coefficient*pi form.
    m = re.search(r"-?\d+(?:[\.,]\d+)?\s*\\?pi", ans, flags=re.I)
    if m:
        return m.group(0).replace(" ", "")

    # Prefer common fraction.
    m = re.search(r"-?\d+(?:[\.,]\d+)?\s*/\s*-?\d+(?:[\.,]\d+)?", ans)
    if m:
        return m.group(0).replace(" ", "")

    # Otherwise last numeric token, with optional thousand separators.
    nums = re.findall(r"-?\d+(?:[\.,]\d+)*", ans)
    if nums:
        return nums[-1]
    return ans.strip()


def extract_final_answer(text: str, fallback_last_number: bool = True) -> Optional[str]:
    text = normalize_unicode_text(text)
    if not text.strip():
        return None
    matches = []
    for pat in ANSWER_ANCHOR_PATTERNS:
        for m in re.finditer(pat, text, flags=re.I):
            matches.append((m.start(), m.group(1)))
    if matches:
        matches.sort(key=lambda x: x[0])
        cand = clean_answer_candidate(matches[-1][1])
        if cand and not any(tok in cand.lower() for tok in BAD_ANSWER_TOKENS):
            return cand
    if fallback_last_number:
        cand = clean_answer_candidate(text)
        if cand and not any(tok in cand.lower() for tok in BAD_ANSWER_TOKENS):
            return cand
    return None


def _normalize_num_token(x: str) -> str:
    x = str(x).strip().replace(" ", "")
    # 40.320 or 1,234,567 => thousand separators.
    if re.fullmatch(r"-?\d{1,3}([\.,]\d{3})+", x):
        return x.replace(".", "").replace(",", "")
    # 4,5 => 4.5 decimal comma.
    if re.fullmatch(r"-?\d+,\d{1,6}", x):
        return x.replace(",", ".")
    return x.replace(",", "")


def parse_numeric_answer(ans: Any) -> Optional[float]:
    if ans is None:
        return None
    s = clean_answer_candidate(str(ans))
    s = normalize_unicode_text(s)
    s = s.replace("$", "").strip()

    # Latex fraction.
    m = re.fullmatch(r"(-?)\\frac\{\s*(-?\d+(?:[\.,]\d+)?)\s*\}\{\s*(-?\d+(?:[\.,]\d+)?)\s*\}", s)
    if m:
        sign = -1.0 if m.group(1) == "-" else 1.0
        a = float(_normalize_num_token(m.group(2)))
        b = float(_normalize_num_token(m.group(3)))
        return sign * a / b if b != 0 else None

    # Common fraction.
    m = re.fullmatch(r"(-?\d+(?:[\.,]\d+)?)\s*/\s*(-?\d+(?:[\.,]\d+)?)", s)
    if m:
        a = float(_normalize_num_token(m.group(1)))
        b = float(_normalize_num_token(m.group(2)))
        return a / b if b != 0 else None

    # pi forms: pi, -pi/2, 36\pi.
    pi_s = s.replace("\\pi", "pi").replace("π", "pi")
    m = re.fullmatch(r"(-?)(?:(\d+(?:[\.,]\d+)?))?\s*pi(?:\s*/\s*(\d+(?:[\.,]\d+)?))?", pi_s, flags=re.I)
    if m:
        sign = -1.0 if m.group(1) == "-" else 1.0
        coef = float(_normalize_num_token(m.group(2))) if m.group(2) else 1.0
        den = float(_normalize_num_token(m.group(3))) if m.group(3) else 1.0
        return sign * coef * math.pi / den if den != 0 else None

    # Plain number.
    if re.fullmatch(r"-?\d+(?:[\.,]\d+)*", s):
        try:
            return float(_normalize_num_token(s))
        except Exception:
            return None
    return None


def relative_error(pred: Any, gold: Any) -> Optional[float]:
    p = parse_numeric_answer(pred)
    g = parse_numeric_answer(gold)
    if p is None or g is None:
        return None
    return abs(p - g) / max(1.0, abs(g))


def score_one(pred: Any, gold: Any) -> int:
    # Exact string fallback for non-scalar answers.
    if pred is not None and gold is not None:
        if clean_answer_candidate(str(pred)) == clean_answer_candidate(str(gold)):
            return 10
    err = relative_error(pred, gold)
    if err is None:
        return 0
    if err <= 0.01:
        return 10
    if err <= 0.10:
        return 5
    if err <= 0.50:
        return 1
    return 0

# Quick sanity checks.
for s in ["Đáp án là: 37", "Câu trả lời là: \\frac{9}{20}", "The answer is: 36\\pi", "#### 40.320"]:
    a = extract_final_answer(s)
    print(s, "=>", a, "=>", parse_numeric_answer(a))


Đáp án là: 37 => 37 => 37.0
Câu trả lời là: \frac{9}{20} => \frac{9}{20} => 0.45
The answer is: 36\pi => 36\pi => 113.09733552923255
#### 40.320 => 40.320 => 40320.0


## Cell 4 — Quality filtering + mixed answer/solution records


In [4]:
def get_query(raw: Dict[str, Any]) -> str:
    return normalize_unicode_text(str(raw.get("query_vi", "") or "")).strip()


def get_response(raw: Dict[str, Any]) -> str:
    return normalize_unicode_text(str(raw.get("response_vi", "") or "")).strip()


def get_type(raw: Dict[str, Any]) -> str:
    return str(raw.get("type", "UNKNOWN") or "UNKNOWN")


def make_answer_prompt(query: str) -> str:
    return f"Câu hỏi: {query}\nĐáp án:"


def make_solution_prompt(query: str) -> str:
    return f"Câu hỏi: {query}\nLời giải:"


def make_prompt(query: str) -> str:
    # Default inference prompt: answer-only. Solution prompt is used separately.
    return make_answer_prompt(query)


def make_answer_target(answer: str) -> str:
    return f" {answer}"


def normalize_query(q: str) -> str:
    q = strip_asy_blocks(q)
    q = normalize_unicode_text(q)
    q = re.sub(r"\s+", " ", q).strip()
    return q


def strip_answer_tail(resp: str) -> str:
    resp = normalize_unicode_text(resp)
    resp = strip_boxed(resp)
    resp = re.sub(r"#+\s*-?\d+(?:[\.,]\d+)?", " ", resp)
    resp = re.split(r"(?:Đáp án là|Câu trả lời là|The answer is|####)\s*[:：]?", resp, flags=re.I)[0]
    resp = resp.replace("$", " ")
    resp = re.sub(r"\s+", " ", resp).strip()
    resp = re.sub(r"^(?:Lời giải|Giải|Bài giải)\s*[:：]\s*", "", resp, flags=re.I).strip()
    return resp


def truncate_solution_text(sol: str, max_chars: int = MAX_SOLUTION_CHARS) -> str:
    sol = re.sub(r"\s+", " ", str(sol or "")).strip()
    if len(sol) <= max_chars:
        return sol
    cut = sol[:max_chars]
    # Prefer ending at a sentence boundary or equation-ish delimiter.
    boundary = max(cut.rfind(". "), cut.rfind("; "), cut.rfind(" nên "), cut.rfind(" Vậy "))
    if boundary >= int(max_chars * 0.55):
        cut = cut[:boundary + 1]
    return cut.strip()


def build_short_solution(resp: str, answer: str) -> str:
    sol = strip_answer_tail(resp)
    sol = truncate_solution_text(sol, MAX_SOLUTION_CHARS)
    if not sol:
        sol = "Từ dữ kiện trong đề, mô hình suy luận để tìm đáp án cuối."
    # Avoid duplicated final-answer anchor inside the reasoning body.
    sol = re.split(r"(?:Đáp án là|Câu trả lời là|The answer is)\s*[:：]?", sol, flags=re.I)[0].strip()
    answer = clean_answer_candidate(answer)
    return f"{sol}\nĐáp án là: {answer}"


def make_solution_target(resp: str, answer: str) -> str:
    return " " + build_short_solution(resp, answer)


def count_numeric_tokens(text: str) -> int:
    return len(re.findall(r"-?\d+(?:[\.,]\d+)?", str(text or "")))


def count_operator_tokens(text: str) -> int:
    return len(re.findall(r"[+\-*/=]", normalize_unicode_text(str(text or ""))))


def type_priority(typ: str) -> int:
    # Prefer cleaner rephrased/answer-augmented data, but keep all types through stratified quotas.
    priority = {
        "GSM_Rephrased": 18,
        "MATH_Rephrased": 17,
        "GSM_AnsAug": 16,
        "MATH_AnsAug": 15,
        "GSM_SV": 11,
        "GSM_FOBAR": 10,
        "MATH_SV": 8,
        "MATH_FOBAR": 8,
    }
    return priority.get(str(typ), 5)


def quality_score(q: str, resp: str, ans: str, typ: str) -> float:
    q_len = len(q)
    r_len = len(resp)
    sol = strip_answer_tail(resp)
    sol_len = len(sol)

    score = 0.0
    score += type_priority(typ)
    score += 16.0 if parse_numeric_answer(ans) is not None else -30.0
    score += 10.0 if re.search(r"(Đáp án là|Câu trả lời là|####|The answer is)", resp, flags=re.I) else -10.0
    score += min(count_numeric_tokens(q), 12) * 1.0
    score += min(count_numeric_tokens(resp), 20) * 0.5
    score += min(count_operator_tokens(resp), 20) * 0.6
    score += 8.0 if 80 <= q_len <= 700 else -8.0
    score += 8.0 if 80 <= sol_len <= MAX_SOLUTION_CHARS else -6.0
    score -= max(0, q_len - 900) / 80.0
    score -= max(0, r_len - 1300) / 100.0

    bad_markers = ["không đủ thông tin", "không xác định", "cannot", "undefined", "nan", "inf"]
    if any(x in resp.lower() for x in bad_markers):
        score -= 60.0
    if "[asy]" in resp.lower() or "[asy]" in q.lower():
        score -= 8.0
    return float(score)


def basic_record_from_raw(raw: Dict[str, Any], i: int, split: str, train_mode: bool = True) -> Tuple[Optional[Dict[str, Any]], Optional[Dict[str, Any]]]:
    q = normalize_query(get_query(raw))
    resp = get_response(raw)
    typ = get_type(raw)
    ans = extract_final_answer(resp, fallback_last_number=False)

    reason = None
    if not q:
        reason = "missing_query"
    elif train_mode and not resp:
        reason = "missing_response"
    elif ans is None or not str(ans).strip():
        reason = "missing_answer"
    elif len(q) > 1400:
        reason = "query_too_long"
    elif any(tok in str(ans).lower() for tok in BAD_ANSWER_TOKENS):
        reason = "bad_answer_token"
    elif train_mode and parse_numeric_answer(ans) is None:
        reason = "non_numeric_answer_for_train"

    if reason:
        return None, {"split": split, "raw_index": i, "reason": reason, "type": typ, "query_vi": q[:300]}

    ans = clean_answer_candidate(ans)
    rec = {
        "id": raw.get("id", i),
        "raw_index": i,
        "type": typ,
        "query_vi": q,
        "response_vi": resp,
        "final_answer": ans,
        "quality_score": quality_score(q, resp, ans, typ),
    }
    return rec, None


def preprocess_base_records(raw_records: List[Dict[str, Any]], split: str, train_mode: bool = True) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    records = []
    drops = []
    seen = set()

    for i, raw in enumerate(raw_records):
        rec, drop = basic_record_from_raw(raw, i, split, train_mode=train_mode)
        if drop is not None:
            drops.append(drop)
            continue

        key = (rec["query_vi"], rec["final_answer"])
        if train_mode and key in seen:
            drops.append({"split": split, "raw_index": i, "reason": "duplicate_query_answer", "type": rec["type"], "query_vi": rec["query_vi"][:300]})
            continue
        seen.add(key)
        records.append(rec)

    return records, drops


def select_quality_train_records(records: List[Dict[str, Any]], max_samples: int, reference_records: Optional[List[Dict[str, Any]]] = None) -> List[Dict[str, Any]]:
    if max_samples is None or len(records) <= max_samples:
        selected = list(records)
        rng = random.Random(SEED)
        rng.shuffle(selected)
        return selected

    ref = reference_records if reference_records else records
    ref_counts = Counter(r["type"] for r in ref)
    total_ref = sum(ref_counts.values()) or 1

    by_type = defaultdict(list)
    for r in records:
        by_type[r["type"]].append(r)

    for typ in by_type:
        # Stable random tie-break without changing quality order too much.
        type_seed = sum((j + 1) * ord(ch) for j, ch in enumerate(str(typ)))
        local_rng = random.Random(SEED + type_seed % 10000)
        by_type[typ].sort(key=lambda r: (-r["quality_score"], local_rng.random()))

    selected = []
    selected_ids = set()

    for typ, cnt in ref_counts.items():
        quota = int(round(max_samples * cnt / total_ref))
        quota = max(1, quota)
        for r in by_type.get(typ, [])[:quota]:
            selected.append(r)
            selected_ids.add((r["raw_index"], r["query_vi"]))

    if len(selected) < max_samples:
        leftovers = [
            r for r in records
            if (r["raw_index"], r["query_vi"]) not in selected_ids
        ]
        leftovers.sort(key=lambda r: -r["quality_score"])
        selected.extend(leftovers[:max_samples - len(selected)])

    selected = selected[:max_samples]
    rng = random.Random(SEED)
    rng.shuffle(selected)
    return selected


def assign_mixed_targets(records: List[Dict[str, Any]], solution_ratio: float = SOLUTION_TARGET_RATIO) -> List[Dict[str, Any]]:
    records = [dict(r) for r in records]
    n_solution = int(round(len(records) * solution_ratio))

    # Pick solution-target examples from records with usable/compact reasoning.
    candidates = []
    for idx, r in enumerate(records):
        sol = strip_answer_tail(r["response_vi"])
        if 80 <= len(sol) <= MAX_SOLUTION_CHARS + 250:
            candidates.append((idx, r["quality_score"], len(sol), r["type"]))

    # Stratify solution examples by type to prevent all reasoning targets coming from one type.
    cand_by_type = defaultdict(list)
    for item in candidates:
        cand_by_type[item[3]].append(item)
    for typ in cand_by_type:
        cand_by_type[typ].sort(key=lambda x: (-x[1], abs(x[2] - 380)))

    target_types = Counter(r["type"] for r in records)
    solution_indices = set()
    for typ, cnt in target_types.items():
        quota = int(round(n_solution * cnt / max(1, len(records))))
        for idx, _, _, _ in cand_by_type.get(typ, [])[:quota]:
            solution_indices.add(idx)

    if len(solution_indices) < n_solution:
        flat = sorted(candidates, key=lambda x: (-x[1], abs(x[2] - 380)))
        for idx, _, _, _ in flat:
            if len(solution_indices) >= n_solution:
                break
            solution_indices.add(idx)

    for idx, r in enumerate(records):
        if idx in solution_indices:
            r["target_mode"] = "short_solution"
            r["prompt"] = make_solution_prompt(r["query_vi"])
            r["target"] = make_solution_target(r["response_vi"], r["final_answer"])
        else:
            r["target_mode"] = "answer_only"
            r["prompt"] = make_answer_prompt(r["query_vi"])
            r["target"] = make_answer_target(r["final_answer"])
        # Do not keep full response in the JSONL training artifact to save disk.
        r.pop("response_vi", None)

    return records


train_base_records, train_drops = preprocess_base_records(raw_train, "train", train_mode=True)
valid_records, valid_drops = preprocess_base_records(raw_valid, "valid", train_mode=False)

selected_train = select_quality_train_records(train_base_records, TRAIN_MAX_SAMPLES, reference_records=valid_records)
train_records = assign_mixed_targets(selected_train, SOLUTION_TARGET_RATIO)

# Build validation prompts as answer-only records for loss/evaluation consistency.
for r in valid_records:
    r["target_mode"] = "answer_only"
    r["prompt"] = make_answer_prompt(r["query_vi"])
    r["target"] = make_answer_target(r["final_answer"])
    r.pop("response_vi", None)

write_jsonl(train_records, PROC_DIR / "train_quality40k_mixed.jsonl")
write_jsonl(valid_records, PROC_DIR / "valid_answer_eval.jsonl")
write_jsonl(train_drops + valid_drops, PROC_DIR / "drop_log.jsonl")

prep_report = {
    "variant": VARIANT_NAME,
    "target_format": "mixed_answer_only_short_solution",
    "selection": {
        "source": "full_train_json",
        "raw_train_after_basic_clean": len(train_base_records),
        "selected_train": len(train_records),
        "train_max_samples": TRAIN_MAX_SAMPLES,
        "solution_target_ratio": SOLUTION_TARGET_RATIO,
    },
    "counts": {
        "raw_train": len(raw_train),
        "raw_valid": len(raw_valid),
        "train_records": len(train_records),
        "valid_records": len(valid_records),
        "train_dropped": len(train_drops),
        "valid_dropped": len(valid_drops),
    },
    "target_mode_distribution": dict(Counter(r["target_mode"] for r in train_records)),
    "train_type_distribution": dict(Counter(r["type"] for r in train_records)),
    "valid_type_distribution": dict(Counter(r["type"] for r in valid_records)),
    "drop_summary": dict(Counter(d["reason"] for d in train_drops + valid_drops)),
    "quality_score_summary": {
        "min": float(np.min([r["quality_score"] for r in train_records])) if train_records else None,
        "mean": float(np.mean([r["quality_score"] for r in train_records])) if train_records else None,
        "max": float(np.max([r["quality_score"] for r in train_records])) if train_records else None,
    },
}
write_json(prep_report, PROC_DIR / "preprocess_report.json")
print(json.dumps(prep_report, ensure_ascii=False, indent=2)[:3000])

pd.DataFrame(train_records[:8])[['type','target_mode','quality_score','query_vi','final_answer','prompt','target']]


{
  "variant": "v7_answer_only_lora_rule_solver",
  "target_format": "answer_only",
  "counts": {
    "raw_train": 95400,
    "raw_valid": 1000,
    "train_records": 57842,
    "valid_records": 997,
    "train_dropped": 37558,
    "valid_dropped": 3
  },
  "train_type_distribution": {
    "GSM_AnsAug": 6307,
    "MATH_FOBAR": 2435,
    "GSM_FOBAR": 7758,
    "GSM_SV": 9198,
    "GSM_Rephrased": 17061,
    "MATH_Rephrased": 8493,
    "MATH_AnsAug": 4307,
    "MATH_SV": 2283
  },
  "valid_type_distribution": {
    "GSM_Rephrased": 197,
    "MATH_Rephrased": 115,
    "MATH_SV": 41,
    "GSM_AnsAug": 209,
    "GSM_SV": 97,
    "GSM_FOBAR": 122,
    "MATH_AnsAug": 171,
    "MATH_FOBAR": 45
  },
  "drop_summary": {
    "duplicate_query_answer": 37460,
    "missing_answer": 95,
    "query_too_long": 6
  }
}


,type,query_vi,final_answer,prompt,target
0,GSM_AnsAug,Một thẩm phán giám sát mười bảy vụ án. Hai ngư...,4,Câu hỏi: Một thẩm phán giám sát mười bảy vụ án...,4
1,MATH_FOBAR,"Giả sử X là thừa số của $a$, $a$ là ước của $1...",3,"Câu hỏi: Giả sử X là thừa số của $a$, $a$ là ư...",3
2,GSM_FOBAR,"Randy đạt 90, 98, x và 94 trong bốn câu hỏi đầ...",92,"Câu hỏi: Randy đạt 90, 98, x và 94 trong bốn c...",92
3,GSM_AnsAug,Kim gọi một bữa ăn giá 10 USD và một đồ uống g...,5,Câu hỏi: Kim gọi một bữa ăn giá 10 USD và một ...,5
4,GSM_SV,"John có x miếng kẹo cao su, Cole có 45 miếng k...",54,"Câu hỏi: John có x miếng kẹo cao su, Cole có 4...",54


## Cell 5 — Load tokenizer/model and build mixed-target torch datasets


In [5]:
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = find_model_path()

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(str(MODEL_PATH), local_files_only=True)

# Force safe eos/pad ids required by the competition note.
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token is not None else "<|endoftext|>"
model.config.pad_token_id = SAFE_EOS_ID
model.config.eos_token_id = SAFE_EOS_ID

print("vocab_size:", len(tokenizer), "pad:", tokenizer.pad_token_id, "eos:", tokenizer.eos_token_id)

class MixedTargetDataset(Dataset):
    def __init__(self, records: List[Dict[str, Any]], tokenizer, max_length: int):
        self.records = records
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        r = self.records[idx]
        prompt = r["prompt"]
        target = r["target"] + (self.tokenizer.eos_token or "")

        prompt_ids = self.tokenizer(prompt, add_special_tokens=False)["input_ids"]
        full = self.tokenizer(
            prompt + target,
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_length,
        )
        input_ids = full["input_ids"]
        attention_mask = full["attention_mask"]
        labels = input_ids.copy()
        prompt_len = min(len(prompt_ids), len(labels))
        labels[:prompt_len] = [-100] * prompt_len

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

class CausalLMCollator:
    def __init__(self, tokenizer, label_pad_token_id: int = -100):
        self.tokenizer = tokenizer
        self.label_pad_token_id = label_pad_token_id

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        for f in features:
            pad_len = max_len - len(f["input_ids"])
            batch["input_ids"].append(torch.cat([f["input_ids"], torch.full((pad_len,), self.tokenizer.pad_token_id, dtype=torch.long)]))
            batch["attention_mask"].append(torch.cat([f["attention_mask"], torch.zeros(pad_len, dtype=torch.long)]))
            batch["labels"].append(torch.cat([f["labels"], torch.full((pad_len,), self.label_pad_token_id, dtype=torch.long)]))
        return {k: torch.stack(v) for k, v in batch.items()}

train_dataset = MixedTargetDataset(train_records, tokenizer, MAX_LENGTH)
eval_loss_records = valid_records[:min(EVAL_LOSS_MAX_SAMPLES, len(valid_records))]
eval_dataset = MixedTargetDataset(eval_loss_records, tokenizer, MAX_LENGTH)
collator = CausalLMCollator(tokenizer)

print("train_dataset:", len(train_dataset), "eval_dataset:", len(eval_dataset))
print("target modes:", Counter(r["target_mode"] for r in train_records))


MODEL_PATH: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
h.{0...11}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab_size: 50258 pad: 50256 eos: 50256
train_dataset: 57842 eval_dataset: 512


## Cell 6 — Apply LoRA

In [6]:

try:
    from peft import LoraConfig, get_peft_model, TaskType
except Exception as e:
    raise ImportError(
        "PEFT is required for this LoRA variant. On Kaggle, attach a dataset/package with peft if needed, still with Internet OFF."
    ) from e

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["c_attn", "c_proj", "c_fc"],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

if torch.cuda.is_available():
    model = model.cuda()


trainable params: 4,718,592 || all params: 129,158,400 || trainable%: 3.6533


## Cell 7 — Fine-tune answer-only LoRA

In [7]:

from transformers import Trainer, TrainingArguments


def build_training_args() -> TrainingArguments:
    raw_kwargs = dict(
        output_dir=str(MODEL_OUT_DIR),
        overwrite_output_dir=True,
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        eval_steps=EVAL_STEPS,
        save_total_limit=2,
        fp16=torch.cuda.is_available(),
        report_to="none",
        dataloader_num_workers=2,
        remove_unused_columns=False,
        load_best_model_at_end=False,
    )
    sig = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" in sig:
        raw_kwargs["eval_strategy"] = "steps"
    elif "evaluation_strategy" in sig:
        raw_kwargs["evaluation_strategy"] = "steps"

    kwargs = {k: v for k, v in raw_kwargs.items() if k in sig}
    dropped = sorted(set(raw_kwargs) - set(kwargs))
    if dropped:
        print("Dropped unsupported TrainingArguments:", dropped)
    return TrainingArguments(**kwargs)


def build_trainer_kwargs(args: TrainingArguments) -> Dict[str, Any]:
    base = dict(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=collator,
    )
    sig = inspect.signature(Trainer.__init__).parameters
    return {k: v for k, v in base.items() if k in sig}

if DO_TRAIN:
    training_args = build_training_args()
    trainer = Trainer(**build_trainer_kwargs(training_args))
    train_result = trainer.train()
    print(train_result)
    trainer.save_model(str(MODEL_OUT_DIR))
    tokenizer.save_pretrained(str(MODEL_OUT_DIR))
    write_json({"train_result": str(train_result), "variant": VARIANT_NAME}, MODEL_OUT_DIR / "train_result.json")
else:
    print("DO_TRAIN=False, skip training")

model.eval()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Dropped unsupported TrainingArguments: ['overwrite_output_dir']


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
700,2.102797,2.082586
1400,1.959173,1.973994
2100,1.825483,1.923762
2800,1.687224,1.878392
3500,1.667151,1.825433
4200,1.570558,1.828835


TrainOutput(global_step=4520, training_loss=1.9427786485283776, metrics={'train_runtime': 6707.5128, 'train_samples_per_second': 43.117, 'train_steps_per_second': 0.674, 'total_flos': 2.142657548780544e+16, 'train_loss': 1.9427786485283776, 'epoch': 5.0})


## Cell 8 — Inference policy


In [8]:
INFERENCE_POLICY = {
    "variant": VARIANT_NAME,
    "model_only": True,
    "no_rule_solver": True,
    "no_python_math_solver_from_query": True,
    "allowed_python_usage": [
        "data_loading",
        "data_cleaning",
        "quality_filtering_without_solving_query",
        "answer_extraction_from_model_output",
        "local_validation_scoring",
        "json_export",
    ],
    "train_target_format": "mixed_answer_only_short_solution",
    "candidate_sources": ["model_answer_greedy", "model_solution_greedy"] + (["model_answer_sample"] if USE_SAMPLING_FALLBACK else []),
}
print(json.dumps(INFERENCE_POLICY, ensure_ascii=False, indent=2))


(None, 'no_rule') -- Susan đang chơi một trò chơi board game có 48 ô. Sau ba lượt, cô ấy đã đi được 1
('19', 'rule_triangle_two_equal_sides') -- Một tam giác có hai cạnh bằng 7 và cạnh còn lại bằng 5. Chu vi tam giác là bao n
('360', 'rule_lcm') -- Tìm bội số chung nhỏ nhất của 24 và 90.
('2', 'rule_distinct_prime_factors') -- 56 có bao nhiêu thừa số nguyên tố phân biệt?


## Cell 9 — Model-only answer + solution generation


In [9]:
@torch.no_grad()
def generate_tail(prompt: str, max_new_tokens: int, do_sample: bool = False, num_return_sequences: int = 1) -> List[str]:
    model.eval()
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(model.device)

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        pad_token_id=SAFE_EOS_ID,
        eos_token_id=SAFE_EOS_ID,
        num_return_sequences=num_return_sequences,
    )
    if do_sample:
        gen_kwargs.update(dict(do_sample=True, temperature=GEN_TEMPERATURE, top_p=GEN_TOP_P))
    else:
        gen_kwargs.update(dict(do_sample=False, num_beams=1))

    out = model.generate(**inputs, **gen_kwargs)
    decoded = tokenizer.batch_decode(out, skip_special_tokens=True)

    tails = []
    for text in decoded:
        if text.startswith(prompt):
            tail = text[len(prompt):]
        else:
            # Fallback split only removes the prompt-ish prefix; it does not solve anything.
            tail = text.split("Lời giải:")[-1] if "Lời giải:" in text else text.split("Đáp án:")[-1]
        tail = normalize_unicode_text(tail).strip()
        if tail:
            tails.append(tail)
    return tails


def generate_answer_candidates(query: str, do_sample: bool = False, num_return_sequences: int = 1) -> List[Dict[str, Any]]:
    prompt = make_answer_prompt(normalize_query(query))
    tails = generate_tail(prompt, ANSWER_GEN_MAX_NEW_TOKENS, do_sample=do_sample, num_return_sequences=num_return_sequences)
    cands = []
    for tail in tails:
        ans = extract_final_answer(tail, fallback_last_number=True)
        if ans is None:
            ans = clean_answer_candidate(tail)
        ans = clean_answer_candidate(ans)
        if ans:
            cands.append({"kind": "answer", "answer": ans, "tail": tail, "prompt": prompt})
    return cands


def generate_solution_candidates(query: str, do_sample: bool = False, num_return_sequences: int = 1) -> List[Dict[str, Any]]:
    prompt = make_solution_prompt(normalize_query(query))
    tails = generate_tail(prompt, SOLUTION_GEN_MAX_NEW_TOKENS, do_sample=do_sample, num_return_sequences=num_return_sequences)
    cands = []
    for tail in tails:
        # The model was prompted after "Lời giải:", so add it back for extractor/format.
        full_solution = "Lời giải: " + tail.strip()
        ans = extract_final_answer(full_solution, fallback_last_number=True)
        ans = clean_answer_candidate(ans) if ans is not None else None
        if ans:
            cands.append({"kind": "solution", "answer": ans, "tail": tail, "full_solution": full_solution, "prompt": prompt})
    return cands


def canonical_answer_key(ans: str) -> str:
    ans = clean_answer_candidate(ans)
    val = parse_numeric_answer(ans)
    if val is not None and math.isfinite(float(val)):
        return f"num:{float(val):.10g}"
    return f"str:{ans}"


def choose_answer_from_candidates(cands: List[Dict[str, Any]]) -> Tuple[Optional[str], Dict[str, Any]]:
    cleaned = []
    for c in cands:
        ans = clean_answer_candidate(c.get("answer", ""))
        if ans:
            cc = dict(c)
            cc["answer"] = ans
            cc["key"] = canonical_answer_key(ans)
            cleaned.append(cc)

    if not cleaned:
        return None, {"source": "none", "candidates": []}

    grouped = defaultdict(list)
    for c in cleaned:
        grouped[c["key"]].append(c)

    # Prefer agreement; if tied, prefer answer-greedy for numeric stability.
    def group_rank(item):
        key, group = item
        has_answer = any(c["kind"] == "answer" for c in group)
        has_solution = any(c["kind"] == "solution" for c in group)
        shortest_ans = min(len(c["answer"]) for c in group)
        return (-len(group), not (has_answer and has_solution), not has_answer, shortest_ans, key)

    best_key, best_group = sorted(grouped.items(), key=group_rank)[0]
    representative = sorted(best_group, key=lambda c: (c["kind"] != "answer", len(c["answer"]), c["answer"]))[0]
    return representative["answer"], {
        "source": "model_vote",
        "candidates": [{"kind": c["kind"], "answer": c["answer"], "key": c["key"], "tail": c.get("tail", "")[:300]} for c in cleaned],
        "canonical_counts": {k: len(v) for k, v in grouped.items()},
        "best_key": best_key,
        "best_kind": representative["kind"],
    }


def sanitize_solution_output(solution_tail: str, final_answer: str) -> str:
    body = normalize_unicode_text(solution_tail or "")
    body = re.sub(r"\s+", " ", body).strip()
    body = re.split(r"(?:Đáp án là|Câu trả lời là|The answer is|####)\s*[:：]?", body, flags=re.I)[0].strip()
    body = body.replace("$", "").strip()
    if not body:
        body = "Mô hình suy luận từ dữ kiện trong đề để tìm đáp án cuối."
    # Limit output length to reduce rambling while preserving final answer.
    body = truncate_solution_text(body, 700)
    return f"Lời giải: {body}\nĐáp án là: {clean_answer_candidate(final_answer)}"


def make_model_output(answer: str, source: str = "model", solution_tail: Optional[str] = None) -> str:
    answer = clean_answer_candidate(answer or "")
    if not answer:
        # Last-resort formatting fallback for invalid generations only.
        # It does not inspect the query and does not compute an answer.
        answer = "0"
    if solution_tail:
        return sanitize_solution_output(solution_tail, answer)
    return f"Lời giải ngắn: Mô hình dự đoán đáp án từ câu hỏi đã cho.\nĐáp án là: {answer}"


def predict_one(query: str) -> Tuple[str, Dict[str, Any]]:
    query = normalize_query(query)

    source_trace = []

    # 1) Deterministic answer-only generation: usually most stable for numeric scoring.
    answer_cands = generate_answer_candidates(query, do_sample=False, num_return_sequences=1)
    source_trace.append({"source": "model_answer_greedy", "candidates": answer_cands})

    # 2) Deterministic short-solution generation: used when it agrees with answer-only,
    # or as fallback if answer-only fails.
    solution_cands = generate_solution_candidates(query, do_sample=False, num_return_sequences=1)
    source_trace.append({"source": "model_solution_greedy", "candidates": solution_cands})

    all_candidates = answer_cands + solution_cands

    # 3) Optional model-only sampling fallback. Disabled by default.
    if USE_SAMPLING_FALLBACK and not answer_cands:
        sample_cands = generate_answer_candidates(query, do_sample=True, num_return_sequences=NUM_SAMPLE_CANDIDATES)
        all_candidates.extend(sample_cands)
        source_trace.append({"source": "model_answer_sample", "candidates": sample_cands})

    ans, info = choose_answer_from_candidates(all_candidates)
    if ans is not None:
        # Prefer a generated solution only if its answer agrees with the selected answer.
        selected_key = canonical_answer_key(ans)
        agreeing_solutions = [
            c for c in solution_cands
            if canonical_answer_key(c["answer"]) == selected_key and c.get("tail")
        ]
        if agreeing_solutions:
            return make_model_output(ans, source="model_solution_agree", solution_tail=agreeing_solutions[0]["tail"]), {
                "source": "model_solution_agree",
                "answer": ans,
                "trace": source_trace,
                **info,
            }

        return make_model_output(ans, source="model_answer"), {
            "source": "model_answer",
            "answer": ans,
            "trace": source_trace,
            **info,
        }

    return make_model_output("0", source="fallback_format_only"), {
        "source": "fallback_format_only",
        "answer": "0",
        "trace": source_trace,
    }


## Cell 10 — Validation inference + report

In [10]:

def evaluate_records(records: List[Dict[str, Any]], max_samples: Optional[int] = None) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    eval_records = list(records)
    if max_samples is not None and len(eval_records) > max_samples:
        rng = random.Random(SEED)
        eval_records = rng.sample(eval_records, max_samples)

    outputs = []
    for i, r in enumerate(eval_records):
        if i % 50 == 0:
            print(f"Evaluating {i}/{len(eval_records)}")
        model_output, info = predict_one(r["query_vi"])
        pred_answer = extract_final_answer(model_output, fallback_last_number=True)
        gold_answer = r["final_answer"]
        sc = score_one(pred_answer, gold_answer)
        err = relative_error(pred_answer, gold_answer)
        outputs.append({
            "id": r.get("id", i),
            "raw_index": r.get("raw_index", i),
            "query_vi": r["query_vi"],
            "type": r["type"],
            "gold_answer": gold_answer,
            "pred_answer": clean_answer_candidate(pred_answer or ""),
            "relative_error": err,
            "score": sc,
            "model_output": model_output,
            "predict_info": info,
        })

    n = len(outputs)
    report = {
        "variant": VARIANT_NAME,
        "n": n,
        "raw_score": int(sum(x["score"] for x in outputs)),
        "score_10": float(sum(x["score"] for x in outputs) / n) if n else 0.0,
        "exact_10_count": int(sum(x["score"] == 10 for x in outputs)),
        "score_5_count": int(sum(x["score"] == 5 for x in outputs)),
        "score_1_count": int(sum(x["score"] == 1 for x in outputs)),
        "score_0_count": int(sum(x["score"] == 0 for x in outputs)),
        "extract_rate": float(np.mean([bool(x["pred_answer"]) for x in outputs])) if n else 0.0,
        "source_counts": dict(Counter(x["predict_info"].get("source", "unknown") for x in outputs)),
        "inference_policy": INFERENCE_POLICY,
    }
    return outputs, report

if DO_VALIDATE:
    valid_outputs, valid_report = evaluate_records(valid_records, VALID_EVAL_MAX_SAMPLES)
    write_json(valid_outputs, WORK_DIR / "valid_output.json")
    write_json(valid_report, WORK_DIR / "valid_report.json")
    print(json.dumps(valid_report, ensure_ascii=False, indent=2))
else:
    valid_outputs, valid_report = [], {}
    print("DO_VALIDATE=False, skip validation")


Evaluating 0/997
Evaluating 50/997
Evaluating 100/997
Evaluating 150/997
Evaluating 200/997
Evaluating 250/997
Evaluating 300/997
Evaluating 350/997
Evaluating 400/997
Evaluating 450/997
Evaluating 500/997
Evaluating 550/997
Evaluating 600/997
Evaluating 650/997
Evaluating 700/997
Evaluating 750/997
Evaluating 800/997
Evaluating 850/997
Evaluating 900/997
Evaluating 950/997
{
  "variant": "v7_answer_only_lora_rule_solver",
  "n": 997,
  "raw_score": 1778,
  "score_10": 1.7833500501504513,
  "exact_10_count": 128,
  "score_5_count": 42,
  "score_1_count": 288,
  "score_0_count": 539,
  "extract_rate": 1.0,
  "source_counts": {
    "model": 972,
    "rule_lcm": 4,
    "rule_floor_sqrt": 2,
    "rule_distinct_prime_factors": 1,
    "rule_average_doubled_middle": 3,
    "rule_direct_arithmetic_expression": 11,
    "rule_gcd": 3,
    "rule_triangle_sum_three_sides": 1
  }
}


## Cell 11 — Report by type and error table

In [11]:

if valid_outputs:
    df_eval = pd.DataFrame(valid_outputs)

    def summarize_group(g: pd.DataFrame) -> pd.Series:
        return pd.Series({
            "n": len(g),
            "score_10": g["score"].sum() / len(g) if len(g) else 0,
            "extract_rate": (g["pred_answer"].astype(str).str.len() > 0).mean() if len(g) else 0,
            "score_10_count": int((g["score"] == 10).sum()),
            "score_5_count": int((g["score"] == 5).sum()),
            "score_1_count": int((g["score"] == 1).sum()),
            "score_0_count": int((g["score"] == 0).sum()),
        })

    type_report = df_eval.groupby("type").apply(summarize_group).reset_index()
    type_report = type_report.sort_values(["score_10", "n"], ascending=[True, False])
    display(type_report)
    type_report.to_csv(WORK_DIR / "valid_report_by_type.csv", index=False, encoding="utf-8-sig")

    debug_cols = ["raw_index", "type", "score", "relative_error", "gold_answer", "pred_answer", "query_vi", "model_output", "predict_info"]
    errors_df = df_eval[df_eval["score"] < 10].sort_values(["score", "relative_error"], ascending=[True, False], na_position="last")
    display(errors_df[debug_cols].head(30))
    errors_df[debug_cols].to_csv(WORK_DIR / "valid_errors.csv", index=False, encoding="utf-8-sig")

    full_summary = {"overall": valid_report, "by_type": type_report.to_dict(orient="records"), "preprocess": prep_report}
    write_json(full_summary, WORK_DIR / "valid_full_summary.json")
else:
    print("No valid outputs to report.")


,type,n,score_10,extract_rate,score_10_count,score_5_count,score_1_count,score_0_count
0,GSM_AnsAug,209.0,1.368421,1.0,18.0,9.0,61.0,121.0
5,MATH_FOBAR,45.0,1.600000,1.0,5.0,2.0,12.0,26.0
4,MATH_AnsAug,171.0,1.643275,1.0,21.0,6.0,41.0,103.0
3,GSM_SV,97.0,1.783505,1.0,13.0,1.0,38.0,45.0
1,GSM_FOBAR,122.0,1.795082,1.0,18.0,0.0,39.0,65.0
6,MATH_Rephrased,115.0,1.965217,1.0,16.0,7.0,31.0,61.0
7,MATH_SV,41.0,2.024390,1.0,6.0,2.0,13.0,20.0
2,GSM_Rephrased,197.0,2.223350,1.0,31.0,15.0,53.0,98.0


,raw_index,type,score,relative_error,gold_answer,pred_answer,query_vi,model_output,predict_info
941,944,MATH_FOBAR,0,407.000000,0,407,Tính tổng bình phương các nghiệm của phương tr...,Lời giải ngắn: Áp dụng quy tắc tính nhanh từ d...,{'source': 'rule_direct_arithmetic_expression'...
558,561,GSM_AnsAug,0,345.666667,3,1040,Johnny đã viết một bài luận khoảng 150 từ. Mad...,Lời giải ngắn: Tính theo dữ kiện trong đề.\nĐá...,"{'source': 'model', 'answer': '1040', 'candida..."
367,368,GSM_FOBAR,0,249.000000,2,500,Một xe đầu kéo có tải trọng 50000 pound. 10% t...,Lời giải ngắn: Tính theo dữ kiện trong đề.\nĐá...,"{'source': 'model', 'answer': '500', 'candidat..."
958,961,GSM_Rephrased,0,181.857143,70,12800,Nếu Samantha ngủ trung bình 8 tiếng mỗi đêm và...,Lời giải ngắn: Tính theo dữ kiện trong đề.\nĐá...,"{'source': 'model', 'answer': '12800', 'candid..."
798,801,MATH_Rephrased,0,73.605263,\frac{38}{35},81,Tính tổng của $\frac{2}{7}$ và $\frac{8}{10}$.,Lời giải ngắn: Tính theo dữ kiện trong đề.\nĐá...,"{'source': 'model', 'answer': '81', 'candidate..."
685,688,GSM_FOBAR,0,49.000000,2,100,Một bà nội trợ đi chợ. Cô ấy đã tiêu x trong s...,Lời giải ngắn: Tính theo dữ kiện trong đề.\nĐá...,"{'source': 'model', 'answer': '100', 'candidat..."
524,527,GSM_AnsAug,0,39.000000,5,200,Một tạp chí có giá 3 USD/tạp chí. Jewel mua 10...,Lời giải ngắn: Tính theo dữ kiện trong đề.\nĐá...,"{'source': 'model', 'answer': '200', 'candidat..."
994,997,GSM_AnsAug,0,39.000000,12,480,Judy sử dụng 10 cây bút chì trong 5 ngày học t...,Lời giải ngắn: Tính theo dữ kiện trong đề.\nĐá...,"{'source': 'model', 'answer': '480', 'candidat..."
882,885,GSM_FOBAR,0,29.000000,5,150,It takes 3 men an hour to complete a job. If t...,Lời giải ngắn: Tính theo dữ kiện trong đề.\nĐá...,"{'source': 'model', 'answer': '150', 'candidat..."
987,990,GSM_AnsAug,0,29.000000,1,30,Uncle Lou was given four bags of peanuts to ea...,Lời giải ngắn: Tính theo dữ kiện trong đề.\nĐá...,"{'source': 'model', 'answer': '30', 'candidate..."


## Cell 12 — Generate `test_predictions.json` when test exists

In [12]:

def build_test_records(raw_test: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    records = []
    for i, raw in enumerate(raw_test):
        q = normalize_query(str(raw.get("query_vi", "") or ""))
        records.append({
            "id": raw.get("id", i),
            "query_vi": q,
            "type": str(raw.get("type", "UNKNOWN") or "UNKNOWN"),
        })
    return records


def predict_test(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    preds = []
    for i, r in enumerate(records):
        if i % 50 == 0:
            print(f"Predicting {i}/{len(records)}")
        model_output, info = predict_one(r["query_vi"])
        preds.append({
            "id": r["id"],
            "query_vi": r["query_vi"],
            "type": r["type"],
            "model_output": model_output,
        })
    return preds

if DO_TEST_PREDICT and TEST_PATH is not None and TEST_PATH.exists():
    print("Found test file:", TEST_PATH)
    raw_test = ensure_list_records(read_json_or_jsonl(TEST_PATH))
    test_records = build_test_records(raw_test)
    test_predictions = predict_test(test_records)
    write_json(test_predictions, WORK_DIR / "test_predictions.json")
    print("Saved:", WORK_DIR / "test_predictions.json", "n=", len(test_predictions))
elif DO_TEST_PREDICT:
    # Smoke-test output format using first 2 valid examples so the notebook always leaves an example file.
    print("No official test.json found. Creating sample_test_predictions_from_valid.json for format check only.")
    sample_records = [
        {"id": r["id"], "query_vi": r["query_vi"], "type": r["type"]}
        for r in valid_records[:2]
    ]
    sample_predictions = predict_test(sample_records)
    write_json(sample_predictions, WORK_DIR / "sample_test_predictions_from_valid.json")
    display(pd.DataFrame(sample_predictions))
else:
    print("DO_TEST_PREDICT=False, skip test prediction")


No official test.json found. Creating sample_test_predictions_from_valid.json for format check only.
Predicting 0/2


,id,query_vi,type,model_output
0,0,Nếu Susan đang chơi một trò chơi cờ bàn có 48 ...,GSM_Rephrased,Lời giải ngắn: Tính theo dữ kiện trong đề.\nĐá...
1,1,"Nếu $\angle PQR = \angle PRQ$, và độ dài của Q...",MATH_Rephrased,Lời giải ngắn: Tính theo dữ kiện trong đề.\nĐá...


## Cell 13 — Final artifact list

In [13]:

print("Important outputs:")
for p in [
    WORK_DIR / "valid_output.json",
    WORK_DIR / "valid_report.json",
    WORK_DIR / "valid_report_by_type.csv",
    WORK_DIR / "valid_errors.csv",
    WORK_DIR / "test_predictions.json",
    WORK_DIR / "sample_test_predictions_from_valid.json",
    MODEL_OUT_DIR,
]:
    print("-", p, "exists=", p.exists())


Important outputs:
- /kaggle/working/valid_output.json exists= True
- /kaggle/working/valid_report.json exists= True
- /kaggle/working/valid_report_by_type.csv exists= True
- /kaggle/working/valid_errors.csv exists= True
- /kaggle/working/test_predictions.json exists= False
- /kaggle/working/sample_test_predictions_from_valid.json exists= True
- /kaggle/working/lora_v7_answer_only_lora_rule_solver exists= True
